**Sample ID**: 363



**Query**: Rename the file "Project Plan.pdf" in the Google Drive folder named "My Project" to "Project Plan 2024.pdf", grant view access to asmith@clarion.com, and draft an email to the same address with the subject "2024 Project Plan Preview" and the body:
"Hi Alice Smith,\nI have given you view access to the file. Please share your feedback."




**DB Type**: Base Case




**Case Description**: The file "Project Plan.pdf" exists in Google Drive folder named "My project". It has to be renamed to "Project Plan 2024.pdf". View access is to be granted to asmith@clarion.com. A new draft email is to be created with subject "2024 Project Plan Preview", addressed to asmith@clarion.com, and includes the updated message body.




**Global/Context Variables**:

- folder_name = "My Project"
- old_file_name = "Project Plan.pdf"
- new_file_name = "Project Plan 2024.pdf"
- email_subject = "2024 Project Plan Preview"
- email_body = "Hi Alice Smith,\nI have given you view access to the file. Please share your feedback."



**APIs**:

- gmail
- gdrive


# Set Up

## Download relevant files

In [ ]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.2"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"✅ Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.2 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.2.zip (ID: 1hKbgYK5K6OypdTCZOtrKxHHUTkNQXlWy)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.2.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.

Generating FC Schemas
✅ Successfully generated 68 FC Schemas to /content/Schemas


## Install Dependencies and Clone Repositories

In [ ]:
!pip install -r /content/APIs/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 40.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 81.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.1/245.1 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.9/443.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.9/168.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.3/187.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.8 MB/s eta 0:

## Import APIs and initiate DBs

In [ ]:
# Import necessary simulation modules for Google Drive and Gmail
import gdrive
import gmail

# Load databases
gdrive.SimulationEngine.db.load_state("/content/DBs/GDriveDefaultDB.json")
gmail.SimulationEngine.db.load_state("/content/DBs/GmailDefaultDB.json")

# Define global variables
folder_name = "My Project"
old_file_name = "Project Plan.pdf"

# Create the drive
drive1 = gdrive.create_file_or_folder(body={"name": folder_name, "mimeType": 'application/vnd.google-apps.folder'})
print(f"Created new folder: {folder_name}")
drive_id = drive1['id']  # Capture the drive ID

# Create the "Project Plan.pdf" file in 'My Project'
new_file = gdrive.create_file_or_folder(body={
    "name": old_file_name,
    "mimeType": "application/pdf",
    "parents": [drive1['id']]
})

file_created = new_file is not None and 'id' in new_file  # Check if file creation was successful
print(f"Created file: {new_file['name']} in 'My project'. File Creation Success: {file_created}")

Created new folder: My Project
Created file: Project Plan.pdf in 'My project'. File Creation Success: True


# Initial Assertion
1. Assert that exactly one google drive folder with the name "My project" exists in the system.
2. Assert that at least one file exists within the Google Drive folder "My project".
3. Assert that only one file named "Project Plan.pdf" exists within the Google Drive folder "My project".
4. Assert that a file named "Project Plan 2024.pdf" does not exist within the Google Drive folder "My project".
5. Assert that no Gmail draft exists with the given subject and body addressed to asmith@clarion.com.

In [ ]:
from Scripts.assertions_utils import *
# Import necessary simulation modules for Google Drive and Gmail
import gdrive
import gmail

# Define constants
folder_name = "My Project"
old_file_name = "Project Plan.pdf"
new_file_name = "Project Plan 2024.pdf"
recipient_email = "asmith@clarion.com"
email_subject = "2024 Project Plan Preview"
email_body = "Hi Alice Smith,\nI have given you view access to the file. Please share your feedback."

# Assert that exactly one Google Drive folder with the name "My Project" exists.
folder_list_response = gdrive.list_user_files(
    q=f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder'",
    supportsAllDrives=True
)
folders = folder_list_response.get("files", [])
assert len(folders) == 1, f"Expected exactly one folder named '{folder_name}'."
my_project_folder_id = folders[0].get("id")

# Assert that at least one file exists within the "My Project" folder.
file_list_response = gdrive.list_user_files(
    q=f"'{my_project_folder_id}' in parents",
    supportsAllDrives=True
)
files_in_folder = file_list_response.get("files", [])
assert len(files_in_folder) >= 1, f"Expected at least one file in folder '{folder_name}', but found {len(files_in_folder)}."

# Assert that only one file named "Project Plan.pdf" exists within the "My Project" folder.
old_file_list_response = gdrive.list_user_files(
    q=f"'{my_project_folder_id}' in parents and name='{old_file_name}'",
    supportsAllDrives=True
)
old_files = old_file_list_response.get("files", [])
assert len(old_files) == 1, (
    f"Expected exactly one file named '{old_file_name}' in folder '{folder_name}', "
    f"but found {len(old_files)}."
)

# Assert that a file named "Project Plan 2024.pdf" does NOT exist within the "My Project" folder.
new_file_list_response = gdrive.list_user_files(
    q=f"'{my_project_folder_id}' in parents and name='{new_file_name}'",
    supportsAllDrives=True
)
new_files = new_file_list_response.get("files", [])
assert len(new_files) == 0, (
    f"Expected no file named '{new_file_name}' in folder '{folder_name}', "
    f"but found {len(new_files)}."
)

# Assert that no Gmail draft exists with the given subject and body addressed to asmith@clarion.com.
draft_list_response = gmail.list_drafts()
drafts = draft_list_response.get("drafts") or []  # safe access per requirement
matching_drafts = []
for draft in drafts:
    draft_get_response = gmail.get_draft(id=draft["id"], format="full")
    if draft_get_response and "message" in draft_get_response:
        msg = draft_get_response["message"]
        if (
            compare_strings(msg.get("subject"), email_subject)
            and compare_strings(msg.get("recipient"), recipient_email)
            and compare_strings(msg.get("body"), email_body)
        ):
            matching_drafts.append(draft_get_response)

assert len(matching_drafts) == 0, (
    f"Expected no drafts with subject '{email_subject}' and body '{email_body}' to "
    f"'{recipient_email}', but found {len(matching_drafts)}."
)


# Action
- Find the drive named "My Project" and get its ID.  
- Look for the file named "Project Plan.pdf" within the "My Project" drive to check if it exists.  
- Rename the file from "Project Plan.pdf" to "Project Plan 2024.pdf".  
- Grant view access to the file for the email address asmith@clarion.com.  
- Create a draft email addressed to asmith@clarion.com with the subject "2024 Project Plan Preview" and the message:  
  "Hi Alice Smith,  
  I have given you view access to the file. Please share your feedback."

In [ ]:
# Import necessary simulation modules for Google Drive and Gmail
import gdrive
import gmail
import base64

# Constants
drive_name = "My Project"  # Name of the Google Drive to search for
old_file_name = "Project Plan.pdf"  # Original file name
new_file_name = "Project Plan 2024.pdf"  # New file name after renaming
recipient_email = "asmith@clarion.com"  # Email address to grant access and send the draft
email_subject = "2024 Project Plan Preview"  # Subject of the draft email
email_body = "Hi Alice Smith,\nI have given you view access to the file. Please share your feedback."  # Email body text
mime_type = "application/vnd.google-apps.folder"
view_permission_role = "viewer"  # Role for granting view access

# Find the drive "My Project"
drive_response = gdrive.list_user_files(q=f"name='{drive_name}'")  # List drives matching the specified name
if not drive_response or not drive_response.get("files")[0].get("mimeType") == mime_type:  # Check if the drive exists
    print(f"Error: Drive folder '{drive_name}' not found.")
else:
    drive_id = drive_response["files"][0]["id"]  # Retrieve the drive ID
    print(f"Drive folder '{drive_name}' found with ID: {drive_id}")

    # Find the file "Project Plan.pdf" in the drive
    # file_list_response = gdrive.list_user_files(driveId=drive_id, q=f"name='{old_file_name}'")  # List files with the specified name
    file_list_response = gdrive.list_user_files(q=f"'{drive_id}' in parents and name='{old_file_name}'")
    if not file_list_response or not file_list_response.get("files"):  # Check if the file exists
        print(f"Error: File '{old_file_name}' not found in drive '{drive_name}'.")
    else:
        file_id = file_list_response["files"][0]["id"]  # Get the file ID
        print(f"File '{old_file_name}' found with ID: {file_id}")

        # Check if the file is already renamed
        existing_files = gdrive.list_user_files(driveId=drive_id, q=f"name='{new_file_name}'")  # Check if the new file name already exists
        if existing_files and existing_files.get("files"):  # If file with new name exists, skip renaming
            print(f"File is already named '{new_file_name}'. No rename needed.")
        else:
            # Rename the file
            try:
                update_body = {"name": new_file_name}  # Prepare the body for file renaming
                update_response = gdrive.update_file_metadata_or_content(fileId=file_id, body=update_body)  # Perform the rename operation
                print(f"Successfully renamed file to '{new_file_name}'.")
            except Exception as e:
                print(f"Error during file rename: {str(e)}")

        # Check if view access is already granted
        try:
            permissions = gdrive.list_permissions(fileId=file_id).get("permissions", [])  # List existing permissions for the file
            # Check if the desired permission is already granted
            if any(perm.get("emailAddress") == recipient_email and perm.get("role") == view_permission_role for perm in permissions):
                print(f"View access to '{recipient_email}' already exists. No permission update needed.")
            else:
                # Grant view access to the file
                permission_body = {
                    "role": view_permission_role,  # Set permission role as reader
                    "type": "user",  # Permission type: user
                    "emailAddress": recipient_email  # Email address to grant access
                }
                permission_response = gdrive.create_permission(fileId=file_id, body=permission_body)  # Create the permission
                print(f"Successfully granted view access to '{recipient_email}'.")
        except Exception as e:
            print(f"Error checking or granting permission: {str(e)}")

        # Check if the email draft already exists
        try:
            drafts = gmail.list_drafts(userId="me").get("drafts", [])  # List existing drafts for the user
            # Check if a draft with the same subject and recipient already exists
            draft_exists = any(
                draft.get("message", {}).get("subject") == email_subject and
                draft.get("message", {}).get("recipient") == recipient_email
                for draft in drafts
            )
            if draft_exists:  # If the draft already exists, print a message
                print(f"Email draft to '{recipient_email}' with subject '{email_subject}' already exists.")
            else:
                # Create an email draft if it doesn't exist
                # raw_message = f"""To: {recipient_email}
                # Subject: {email_subject}"""

                # Construct a properly formatted RFC 2822 email message
                raw_message = f"""To: {recipient_email}
                Subject: {email_subject}
                MIME-Version: 1.0
                Content-Type: text/plain; charset=utf-8

                {email_body}"""

                # Encode the message in base64url (required by Gmail API)
                raw_message_bytes = raw_message.encode("utf-8")
                raw_message_b64 = base64.urlsafe_b64encode(raw_message_bytes).decode("utf-8")
                message = {
                    "raw": raw_message,  # Set raw message content
                    "recipient": recipient_email,  # Specify recipient email
                    "subject": email_subject,  # Set subject
                    "body": email_body,  # Set message body
                    "labelIds": ["DRAFT"]  # Mark the message as a draft
                }
                draft_body = {"message": message}  # Prepare the draft body
                draft_response = gmail.create_draft(userId="me", draft=draft_body)  # Create the draft
                print("Successfully created email draft.")
        except Exception as e:
            print(f"Error creating or checking email draft: {str(e)}")


Drive folder 'My Project' found with ID: file_3
File 'Project Plan.pdf' found with ID: file_4
Successfully renamed file to 'Project Plan 2024.pdf'.
Successfully granted view access to 'asmith@clarion.com'.
Successfully created email draft.


# Final Assertion
1. Assert that file 'Project Plan.pdf' does not exist in "My project" folder.
2. Assert that old file is not in trash (would indicate copy-delete strategy).
3. Assert that only one file "Project Plan 2024.pdf" exists within the Google Drive folder "My project".
4. Assert that asmith@clarion.com has view access to file named "Project Plan 2024.pdf".
5. Assert that exactly one Gmail draft exists with the given subject and body addressed to asmith@clarion.com.

In [ ]:
from Scripts.assertions_utils import *
import gdrive
import gmail

# Define constants
folder_name = "My Project"
old_file_name = "Project Plan.pdf"
new_file_name = "Project Plan 2024.pdf"
recipient_email = "asmith@clarion.com"
email_subject = "2024 Project Plan Preview"
email_body = "Hi Alice Smith,\nI have given you view access to the file. Please share your feedback."
view_permission_role = "viewer"

# --- Locate the "My Project" folder (exactly one, and it's a folder) ---
folder_list_response = gdrive.list_user_files(q=f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder'", supportsAllDrives=True)
folders = folder_list_response.get("files", [])
assert len(folders) == 1, f"Expected exactly one folder named '{folder_name}'. Found {len(folders)}."
my_project_folder_id = folders[0].get("id")

# --- 1) Assert old file does NOT exist inside the "My Project" folder ---
old_in_folder_resp = gdrive.list_user_files(
    q=f"'{my_project_folder_id}' in parents and name='{old_file_name}'",
    supportsAllDrives=True
)
old_in_folder = old_in_folder_resp.get("files", [])
assert len(old_in_folder) == 0, (
    f"Expected no file named '{old_file_name}' in folder '{folder_name}', "
    f"but found {len(old_in_folder)}."
)

# --- Extra: old file not in trash (copy-delete strategy check) ---
try:
    trashed_files = gdrive.list_user_files(q=f"name='{old_file_name}' and trashed=true", supportsAllDrives=True)
    trashed_count = len(trashed_files.get("files", []))
    assert trashed_count == 0, (
        f"Found {trashed_count} file(s) named '{old_file_name}' in trash - "
        f"indicates copy-delete rather than true rename"
    )
except Exception as e:
    assert False, f"Failed to check trash for old file: {str(e)}"

# --- 2) Assert exactly one "Project Plan 2024.pdf" exists INSIDE the "My Project" folder ---
new_in_folder_resp = gdrive.list_user_files(
    q=f"'{my_project_folder_id}' in parents and name='{new_file_name}'",
    supportsAllDrives=True
)
new_in_folder = new_in_folder_resp.get("files", [])
assert len(new_in_folder) == 1, (
    f"Expected exactly one file named '{new_file_name}' in folder '{folder_name}', "
    f"but found {len(new_in_folder)}."
)
renamed_file_id = new_in_folder[0].get("id")

# --- 3) Assert asmith@clarion.com has viewer access to the specific file ---
permission_list_response = gdrive.list_permissions(fileId=renamed_file_id)
permissions = permission_list_response.get("permissions", [])
view_access_found = False
for permission in permissions:
    if compare_strings(permission.get("emailAddress"), recipient_email) and compare_strings(permission.get("role"), view_permission_role):
        view_access_found = True
        break
assert view_access_found, (
    f"Expected '{recipient_email}' to have '{view_permission_role}' access to file "
    f"'{new_file_name}', but not found."
)

# --- 4) Assert exactly one Gmail draft with the given subject/body/recipient ---
draft_list_response = gmail.list_drafts()
drafts = draft_list_response.get("drafts") or []
matching_drafts = []
for draft in drafts:
    draft_get_response = gmail.get_draft(id=draft["id"], format="full")
    if draft_get_response and "message" in draft_get_response:
        msg = draft_get_response["message"]
        if (
            compare_strings(msg.get("subject", ""), email_subject)
            and compare_strings(msg.get("recipient", ""), recipient_email)
            and compare_strings(msg.get("body", ""), email_body)
        ):
            matching_drafts.append(draft_get_response)

assert len(matching_drafts) == 1, (
    f"Expected 1 draft with subject '{email_subject}' and body '{email_body}' to "
    f"'{recipient_email}', but found {len(matching_drafts)}."
)